In [ ]:
# Step 1: Imports and Setup
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
# Step 2: Define Data Paths
base_dir = "../../../project_datasets/tremor/Tremor_dataset"

movement_dir = os.path.join(base_dir, "preprocessed/movement")
questionnaire_dir = os.path.join(base_dir, "preprocessed/questionnaire")
patients_dir = os.path.join(base_dir, "patients")
file_list_path = os.path.join(base_dir, "preprocessed/file_list.csv")

# Load patient list
file_list_df = pd.read_csv(file_list_path)
print(f"✅ Total subjects listed: {len(file_list_df)}")
display(file_list_df.head())


In [ ]:
# Step 3: Custom Dataset with sequence padding
class ParkinsonsDataset(Dataset):
    def __init__(self, file_list_df, movement_dir, questionnaire_dir, patients_dir, seq_len=3000):
        self.samples = []
        self.seq_len = seq_len

        for _, row in tqdm(file_list_df.iterrows(), total=len(file_list_df)):
            subject_id = str(row['id']).zfill(3)
            movement_path = os.path.join(movement_dir, f"{subject_id}_ml.bin")
            if not os.path.exists(movement_path):
                continue

            try:
                movement_data = np.fromfile(movement_path, dtype=np.float32).reshape(-1, 3)
                # Pad or truncate to fixed sequence length
                if movement_data.shape[0] >= seq_len:
                    movement_data = movement_data[:seq_len]
                else:
                    pad_len = seq_len - movement_data.shape[0]
                    movement_data = np.pad(movement_data, ((0, pad_len), (0, 0)), mode='constant')
            except:
                continue

            # Load questionnaire
            questionnaire_path = os.path.join(questionnaire_dir, f"{subject_id}_ml.bin")
            questionnaire_data = np.zeros(30, dtype=np.float32)
            if os.path.exists(questionnaire_path):
                try:
                    questionnaire_data = np.fromfile(questionnaire_path, dtype=np.float32)
                except:
                    pass

            # Load metadata
            patient_path = os.path.join(patients_dir, f"patient_{subject_id}.json")
            if os.path.exists(patient_path):
                with open(patient_path) as f:
                    patient = json.load(f)
                age = float(patient.get("age", 0))
                weight = float(patient.get("weight", 0))
                height = float(patient.get("height", 0))
                gender = 1 if patient.get("gender") == "male" else 0
                label = 0 if patient.get("condition") == "Healthy" else 1
            else:
                age, weight, height, gender, label = 0, 0, 0, 0, 0

            metadata = np.array([age, weight, height, gender], dtype=np.float32)

            self.samples.append({
                "movement": torch.tensor(movement_data, dtype=torch.float32),
                "questionnaire": torch.tensor(questionnaire_data, dtype=torch.float32),
                "metadata": torch.tensor(metadata, dtype=torch.float32),
                "label": torch.tensor(label, dtype=torch.float32)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [ ]:
dataset = ParkinsonsDataset(
    file_list_df=file_list_df,
    movement_dir=movement_dir,
    questionnaire_dir=questionnaire_dir,
    patients_dir=patients_dir,
    seq_len=3000
)


In [ ]:
print("Dataset size:", len(dataset))


In [ ]:
sample = dataset[0]

print("Keys:", sample.keys())
print("Movement shape:", sample["movement"].shape)
print("Questionnaire shape:", sample["questionnaire"].shape)
print("Metadata:", sample["metadata"])
print("Label:", sample["label"])


In [ ]:
print("First 5 movement rows:\n", sample["movement"][:5])
print("Last 5 movement rows:\n", sample["movement"][-5:])


In [ ]:
# Step 4: CNN + BiLSTM + LayerNorm Model
class ParkinsonsNet(nn.Module):
    def __init__(self, seq_len=3000, movement_input_dim=3, questionnaire_dim=30, metadata_dim=4):
        super(ParkinsonsNet, self).__init__()

        # Movement Branch: 1D CNN + BiLSTM + LayerNorm
        self.cnn = nn.Sequential(
            nn.Conv1d(movement_input_dim, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True, bidirectional=True)
        self.ln_movement = nn.LayerNorm(128)
        self.dropout_movement = nn.Dropout(0.3)

        # Questionnaire MLP
        self.questionnaire_branch = nn.Sequential(
            nn.Linear(questionnaire_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Metadata MLP
        self.metadata_branch = nn.Sequential(
            nn.Linear(metadata_dim, 16),
            nn.ReLU()
        )

        # Final Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32 + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, movement, questionnaire, metadata):
        x = movement.permute(0, 2, 1)
        x = self.cnn(x)  # [B, 32, T//2]
        x = x.permute(0, 2, 1)  # [B, T//2, 32]

        _, (h_n, _) = self.lstm(x)
        lstm_out = torch.cat([h_n[0], h_n[1]], dim=-1)  # [B, 128]
        lstm_out = self.ln_movement(lstm_out)
        lstm_out = self.dropout_movement(lstm_out)

        q_feat = self.questionnaire_branch(questionnaire)  # [B, 32]
        m_feat = self.metadata_branch(metadata)            # [B, 16]

        combined = torch.cat([lstm_out, q_feat, m_feat], dim=1)
        return self.classifier(combined).squeeze(-1)


In [ ]:
# Step 5: Prepare DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load full dataset
full_dataset = ParkinsonsDataset(file_list_df, movement_dir, questionnaire_dir, patients_dir, seq_len=3000)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

# Create loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Step 6: Training Setup
model = ParkinsonsNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        movement = batch['movement'].to(device)
        questionnaire = batch['questionnaire'].to(device)
        metadata = batch['metadata'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(movement, questionnaire, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    preds_list, labels_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            movement = batch['movement'].to(device)
            questionnaire = batch['questionnaire'].to(device)
            metadata = batch['metadata'].to(device)
            labels = batch['label'].to(device)

            outputs = model(movement, questionnaire, metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = (np.array(preds_list) == np.array(labels_list)).mean()
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")


In [ ]:
# Step 7: Final Evaluation on Test Set
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

model.eval()
test_preds = []
test_probs = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        movement = batch['movement'].to(device)
        questionnaire = batch['questionnaire'].to(device)
        metadata = batch['metadata'].to(device)
        labels = batch['label'].to(device)

        outputs = model(movement, questionnaire, metadata)
        probs = torch.sigmoid(outputs)

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend((probs > 0.5).cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
test_probs = np.array(test_probs)
test_preds = np.array(test_preds).flatten()
test_labels = np.array(test_labels)

# Classification Report
print("\n--- Test Classification Report ---")
print(classification_report(test_labels, test_preds, target_names=["Healthy", "Parkinson's"]))

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Healthy", "Parkinson's"], yticklabels=["Healthy", "Parkinson's"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# AUC Score
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC Score: {auc:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Step 8.1: ROC Curve
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
roc_auc = roc_auc_score(test_labels, test_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}", color="darkorange", lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# Step 8.2: Performance Metrics Table
metrics = {
    "Accuracy": accuracy_score(test_labels, test_preds),
    "Precision": precision_score(test_labels, test_preds),
    "Recall": recall_score(test_labels, test_preds),
    "F1-Score": f1_score(test_labels, test_preds),
    "AUC": roc_auc
}

metrics_df = pd.DataFrame(metrics, index=["Score"]).T
styled_df = metrics_df.style.background_gradient(cmap='coolwarm').format("{:.4f}")
display(styled_df)
